# Ablation Study — Loss Function Contribution Analysis

**NCCT Stroke Segmentation V8 — Physics-Informed Loss Functions**

Measures the individual and joint contribution of each auxiliary loss component
using a single fixed architecture (UNet with ResNet34 encoder, ImageNet pretrained).

### Experimental Design

**Fixed architecture:** UNet (ResNet34) — 1 architecture across all runs.

**10 configurations** across 3 random seeds (42, 43, 44):

| Group | Config | ACE λ | Boundary λ | NWU λ | Sym λ |
|---|---|---|---|---|---|
| Add-one-in | **Baseline** | 0 | 0 | 0 | 0 |
| | **+ACE** | 0.01 | 0 | 0 | 0 |
| | **+Boundary** | 0 | 0.1 | 0 | 0 |
| | **+NWU** | 0 | 0 | 0.01 | 0 |
| | **+Symmetry** | 0 | 0 | 0 | 0.05 |
| | **Full** | 0.01 | 0.1 | 0.01 | 0.05 |
| Leave-one-out | **Full − ACE** | 0 | 0.1 | 0.01 | 0.05 |
| | **Full − Boundary** | 0.01 | 0 | 0.01 | 0.05 |
| | **Full − NWU** | 0.01 | 0.1 | 0 | 0.05 |
| | **Full − Symmetry** | 0.01 | 0.1 | 0.01 | 0 |

**Metrics:** Dice, IoU, Sensitivity, Specificity, Precision, HD95

**Total training runs:** 10 configs × 3 seeds = 30 runs

**Reproducibility:** Comprehensive seed management (PyTorch, NumPy, Python, CuDNN).

### 1. Setup

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import os, sys, json, time, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Project modules
sys.path.insert(0, os.getcwd())
from v8_dataset import get_dataloaders_v8
from v8_ablation import (
    set_seed,
    run_ablation,
    print_ablation_table,
    plot_ablation_results,
    plot_training_curves,
    aggregate_results,
    ABLATION_CONFIGS,
    ABLATION_GROUPS,
)

print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Download dataset if not already present
import gdown, zipfile

if not os.path.exists('dataset/train'):
    file_id = '1o0b6Nqs89zYoyRcnih5oGS7sk0bIrImo'
    url = f'https://drive.google.com/uc?id={file_id}'
    output_zip = 'dataset.zip'
    print("Downloading dataset...")
    gdown.download(url, output_zip, quiet=False)
    print("Extracting...")
    os.makedirs('dataset', exist_ok=True)
    with zipfile.ZipFile(output_zip, 'r') as zf:
        zf.extractall('dataset')
    print("Done.")
else:
    print("Dataset already present.")

### 2. Ablation Configurations

Defined in `v8_ablation.ABLATION_CONFIGS`. There are 10 configurations in 2 groups:
- **Add-one-in** (configs 1–6): measures marginal contribution of each loss
- **Leave-one-out** (configs 7–10): measures synergy/redundancy in the full model

In [ ]:
print(f"Total configurations: {len(ABLATION_CONFIGS)}")
print(f"Add-one-in:  {ABLATION_GROUPS['Add-one-in']}")
print(f"Leave-one-out: {ABLATION_GROUPS['Leave-one-out']}")
print()
for name, cfg in ABLATION_CONFIGS.items():
    active = [k.split('_')[1].upper() for k, v in cfg.items() if v > 0]
    print(f"  {name:<20}  λ={[v for v in cfg.values() if v > 0] or '—'}")

### 3. Run Ablation Study

Trains each configuration from scratch with controlled seeds.
Tracks validation metrics and saves the best model state per run.

In [ ]:
# ── Run all 10 configs with 3 seeds each ───────────────────────
# For a quick test (1 seed), set seeds=[42]
# For the full experiment (3 seeds), use seeds=[42, 43, 44]

N_SEEDS = 1  # ← Change to 3 for full experiment
seeds = [42, 43, 44][:N_SEEDS]
configs_to_run = ABLATION_GROUPS['All']  # all 10 configs

print(f"Running {len(configs_to_run)} configs x {len(seeds)} seed(s) = {len(configs_to_run)*len(seeds)} total runs")
print(f"Seeds: {seeds}")
print(f"Configs: {configs_to_run}")
print()

t_start = time.time()

results = run_ablation(
    architecture='unet_resnet34',
    seeds=seeds,
    config_names=configs_to_run,
    epochs=50,
    patience=10,
    batch_size=4,
    verbose=True,
    run_test=True,
)

elapsed = time.time() - t_start
print(f"\nTotal time: {elapsed/60:.1f} minutes ({elapsed/len(seeds)/len(configs_to_run):.1f} min/run)")

### 4. Results — Validation Set

In [ ]:
print('=' * 80)
print('  ABLATION RESULTS — VALIDATION SET')
print('=' * 80)
print_ablation_table(results, metric_key='val_metrics', sort_by='dice')

### 5. Results — Test Set

In [ ]:
print('=' * 80)
print('  ABLATION RESULTS — TEST SET')
print('=' * 80)
print_ablation_table(results, metric_key='test_metrics', sort_by='dice')

### 6. Visualization

Grouped bar charts showing mean ± std across seeds for each configuration.
Red bars = Baseline, Green bars = Full.

In [ ]:
plot_ablation_results(
    results,
    metric_key='val_metrics',
    title='Loss Ablation — Validation Set',
)

In [ ]:
plot_ablation_results(
    results,
    metric_key='test_metrics',
    title='Loss Ablation — Test Set',
)

In [ ]:
# Training curves for add-one-in configs (seed=42)
plot_training_curves(results, config_names=ABLATION_GROUPS['Add-one-in'], seed=42)

### 7. Summary & Interpretation

Key questions the ablation answers:

| Question | How to read |
|---|---|
| Does each loss help individually? | Add-one-in: compare each config to Baseline |
| Which loss contributes most? | Add-one-in: largest Δ from Baseline |
| Are any losses redundant? | Leave-one-out: if removing X from Full doesn't hurt, X is redundant |
| Do losses synergize? | Full > sum of individual gains = synergy |

**Interpretation guide:**
- **Dice / IoU**: Overall segmentation quality
- **HD95**: Boundary precision (lower is better) — critical for stroke
- **Sensitivity**: Lesion detection rate (avoid false negatives)
- **Precision**: Avoid false positives
- **Specificity**: Background identification

In [ ]:
# ── Save results to JSON for later analysis ────────────────
def serialize_results(r):
    """Convert results dict to JSON-serializable format."""
    out = {}
    for cfg_name, seed_dict in r.items():
        out[cfg_name] = {}
        for seed, data in seed_dict.items():
            out[cfg_name][str(seed)] = {}
            for split in ['val_metrics', 'test_metrics']:
                if data.get(split):
                    out[cfg_name][str(seed)][split] = {
                        k: {'mean': v['mean'], 'std': v['std']}
                        for k, v in data[split].items()
                    }
    return out

save_path = 'ablation_results.json'
with open(save_path, 'w') as f:
    json.dump(serialize_results(results), f, indent=2)
print(f"Results saved to {save_path}")

In [ ]:
# ── Quick summary statistics ────────────────────────────────
print('=' * 70)
print('  ABLATION SUMMARY')
print('=' * 70)

for split_name, split_key in [('Validation', 'val_metrics'), ('Test', 'test_metrics')]:
    dice_agg = aggregate_results(results, metric_key=split_key, target='dice')
    hd95_agg = aggregate_results(results, metric_key=split_key, target='hd95')

    print(f"\n{'─' * 50}")
    print(f"  [{split_name}]")
    print(f"{'─' * 50}")

    if 'Baseline' in dice_agg and 'Full' in dice_agg:
        b = dice_agg['Baseline']
        f = dice_agg['Full']
        delta = (f['mean'] - b['mean']) * 100
        print(f"  Full  vs Baseline Dice: {f['mean']*100:.2f}% vs {b['mean']*100:.2f}%  (Δ = {delta:+.2f}%)")

    # Best config by Dice
    best_cfg = max(dice_agg, key=lambda c: dice_agg[c]['mean'])
    print(f"  Best config by Dice: {best_cfg} ({dice_agg[best_cfg]['mean']*100:.2f}%)")

    # Best config by HD95
    valid_hd95 = {c: v for c, v in hd95_agg.items() if not np.isnan(v['mean'])}
    if valid_hd95:
        best_hd95 = min(valid_hd95, key=lambda c: valid_hd95[c]['mean'])
        print(f"  Best config by HD95: {best_hd95} ({hd95_agg[best_hd95]['mean']:.2f} px)")

print(f"\n{'=' * 70}")